# Cash-flow forecast: set up the project

This notebook sets up the cash-flow forecasting project in this (empty) Dataiku project, the way it runs in
production. It reads the 4 curated tables the warehouse team builds from the 17 detailed ones (client, monthly
flows by origin, scheduled payments, monthly balances), through the read-only `EDW_AI` connection (table and
column names: `python/warehouse.py` in the project library), and builds:

- **Raw data**: the 4 curated warehouse tables and the load record, read-only.
- **Staging**: one SQL recipe per table that gives back the standard column names, and a load check
  that stops everything if the latest load is incomplete. The load also sets the forecast's cut-off month.
- **Data prep**: the category rules (an editable dataset), monthly flows by category, obligations, and the model
  rows.
- **Training (code)** and **Training (visual)**: the same rows trained two ways (Python recipes with
  CatBoost; visual ML, tracked by Dataiku), each tested on the last 6 months.
- **Inference**: the 3-month forecast from either (project variable `model_source`), plus the obligations.
- Two scenarios: **Monthly forecast** (after each load) and **Retrain and test the models**.

It then trains and tests the models once and builds the first forecast.

**How to use it:** create an empty project, import this notebook (Notebooks > New notebook > Import), check the
settings below, then run all cells (Cell > Run All): about 15 minutes the first time (creating the code env and
training). It only sets up an empty project: after that, work in the flow and run the scenarios.

In the sandbox, the warehouse tables come from the notebook `warehouse_simulator.ipynb` (run it first).

In [ ]:
EDW_CONNECTION = "EDW_AI"             # read-only connection to the warehouse tables built by the warehouse team
EDW_SCHEMA = "edw_ai"                 # schema of those tables
CONNECTION = "postgres_local"         # where the project writes its own datasets (a SQL connection)
FOLDER_CONNECTION = "filesystem_folders"   # where the code models are saved
CODE_ENV = "cashflow_forecast"        # Python code env for the recipes (created below with CatBoost if missing)
MODEL_SOURCE = "code"                 # which models feed final_forecast: "code" (CatBoost recipes) or "visual"

## 1. Code environment

The Python recipes need CatBoost. This creates a code env with it, unless it already exists. If you are not
allowed to create code envs, ask an admin, or set `CODE_ENV` above to an existing env that has CatBoost. The
visual models use Dataiku's built-in Python.

In [ ]:
import dataiku

client = dataiku.api_client()
project = client.get_project(dataiku.default_project_key())


def ensure_code_env(name, packages=("catboost>=1.2", "scikit-learn")):
    """Create a Python code env with CatBoost, unless one with this name already exists (takes a few minutes)."""
    if name in [e["envName"] for e in client.list_code_envs()]:
        print(f"Code env {name} already exists")
        return
    error = None
    for interpreter in ("PYTHON312", "PYTHON311", "PYTHON310", "PYTHON39"):
        try:
            client.create_code_env("PYTHON", name, "DESIGN_MANAGED", {
                "pythonInterpreter": interpreter, "installCorePackages": True, "installJupyterSupport": False})
            break
        except Exception as e:          # interpreter not installed on this DSS: try the next one
            error = e
    else:
        raise RuntimeError(f"Could not create code env {name}: {error}")
    env = client.get_code_env("PYTHON", name)
    settings = env.get_settings()
    settings.set_required_packages(*packages)
    settings.save()
    env.update_packages()
    print(f"Code env {name} created ({interpreter}) with {', '.join(packages)}")


ensure_code_env(CODE_ENV)

## 2. Code

The project library gets two files: `warehouse.py` (the warehouse tables and their column names) and
`cashflow.py` (categories and their default rules, the inputs the models see, how the code models are trained).
Each recipe below becomes a Python recipe in the flow.

In [ ]:
WAREHOUSE_CODE = r'''
"""Warehouse tables (project library: python/warehouse.py).

In the schema read through the EDW_AI connection, the warehouse team builds the 17 detailed tables of the data
requirements (DETAIL_TABLES) and, from them, 4 curated tables (TABLES) plus a load record. The project reads
only the curated tables; the detailed ones stay in the warehouse for checks and drill-down.
Table names are CFF_ + the source table's name where it is known; columns follow the EDW style seen in
FCT_PAYROLL_MSTR (CSTMR_CIF, ACNT_NMBR, TOT_AMT, ..._DT, ..._FLG with Y / N). These names are our proposal until
the warehouse team confirms them: if they differ, only this file changes.
The project's staging recipes rename every column back to the standard names used by the recipes.
"""

# curated tables the project reads. standard name: (warehouse table, what the warehouse team builds it from)
TABLES = {
    "clients": ("CFF_CLIENT", "detailed tables 1, 2, 16"),
    "monthly_flows_by_origin": ("CFF_CLIENT_MNTHLY_FLOW", "detailed table 3, and CFF_SCHEDULED_PYMT"),
    "scheduled_payments": ("CFF_SCHEDULED_PYMT", "detailed tables 5, 7, 8, 9, 11, 13, 17"),
    "monthly_balances": ("CFF_CLIENT_MNTHLY_BAL", "detailed tables 2, 4"),
}
COLUMNS = {
    "clients": ["client_no", "party_type", "segment", "sub_segment", "sector", "industry", "region", "city",
                "joining_date", "legal_form", "sole_proprietor_flag", "last_annual_sales", "number_of_employees",
                "parent_client_no", "status", "closure_date", "current_accounts", "restricted_accounts"],
    # one row per client, month, direction, origin type and transaction type tag. Leaves out transfers between the
    # client's own accounts and reversed postings. known_1..3: the part of the amount that came from scheduled
    # payments known 1, 2 or 3 months before the month
    "monthly_flows_by_origin": ["client_no", "month", "direction", "origin_type", "txn_type_tag", "amount", "postings",
                                "known_1", "known_2", "known_3"],
    # one row per scheduled payment, past and future; known_date: when the schedule existed
    "scheduled_payments": ["origin_type", "origin_ref", "client_no", "direction", "txn_type_tag", "due_date", "amount",
                           "known_date"],
    "monthly_balances": ["client_no", "month", "balance_end", "overdraft_limit"],
}

# the 17 detailed tables. standard name: (warehouse table, where the warehouse team takes it from)
DETAIL_TABLES = {
    "clients": ("CFF_CBG_CUSTOMER", "EDW_DM.DM_CBG_CUSTOMER"),
    "accounts": ("CFF_ACNT", "to find"),
    "transactions": ("CFF_STATEMENT_ENTRY", "EDW.FCT_STATEMENT_ENTRY"),
    "daily_balances": ("CFF_ACNT_DAILY_BAL", "to find"),
    "standing_orders": ("CFF_STANDING_ORDER", "to find"),
    "payroll_files": ("CFF_PAYROLL_MSTR", "EDW.FCT_PAYROLL_MSTR"),
    "financing_schedules": ("CFF_LOAN_FUTURE_BALANCE", "EDW.FCT_LOAN_FUTURE_BALANCE + EDW.DIM_LOAN"),
    "deposits": ("CFF_DPST", "to find"),
    "sadad_payments": ("CFF_SADAD_TRNS", "EDW.FCT_SADAD_TRNS"),
    "code_lists": ("CFF_CODE_LIST", "various"),
    "trade_finance": ("CFF_TRADE_FINANCE", "to find"),
    "card_sales": ("CFF_MRCHNT_STLMNT", "to find"),
    "corporate_cards": ("CFF_CORP_CARD", "to find"),
    "bills_issued": ("CFF_BILL_PAYMENT", "EDW.DIM_BUS_BL_ACNT + EDW.FCT_BILL_PAYMENT"),
    "client_monthly_summary": ("CFF_CBG_CUSTOMER_MNTHLY", "EDW_DM.DM_CBG_CUSTOMER"),
    "group_structure": ("CFF_CSTMR_GROUP", "to find"),
    "cheques": ("CFF_CHQ", "to find"),
}
# one row per table per load: LOAD_ID, LOAD_DTTS, AS_OF_DT, SRC_TBL_NM, TRGT_TBL_NM, ROW_CNT, LOAD_STA_CD
LOAD_RECORD = "CFF_LOAD_RECORD"
AUDIT_COLUMNS = ["ETL_LOAD_ID", "ETL_LOAD_DTTS"]      # added by the warehouse team to every table

DETAIL_COLUMNS = {
    "clients": ["client_no", "party_type", "segment", "sub_segment", "sector", "industry", "region", "city",
                "joining_date", "legal_form", "sole_proprietor_flag", "last_annual_sales", "number_of_employees",
                "parent_client_no", "status", "closure_date"],
    "accounts": ["account_no", "client_no", "product_type", "product_code", "currency", "open_date", "close_date",
                 "status", "restricted_flag", "payroll_account_flag"],
    "transactions": ["txn_id", "account_no", "posting_date", "value_date", "amount_sar", "amount_orig", "currency",
                     "exchange_rate", "debit_credit", "txn_code", "channel", "txn_type_tag", "counterparty_iban_code",
                     "counterparty_bank", "counterparty_name", "own_account_flag", "origin_type", "origin_ref",
                     "reversal_flag", "original_ref"],
    "daily_balances": ["account_no", "balance_date", "ledger_balance", "available_balance", "blocked_amount",
                       "hold_amount", "overdraft_limit"],
    "standing_orders": ["instruction_id", "account_no", "beneficiary_type", "beneficiary_iban_code", "amount",
                        "currency", "frequency", "start_date", "end_date", "next_payment_date", "status",
                        "created_date", "changed_date"],
    "payroll_files": ["FILE_ID", "DBT_CSTMR_CIF", "PAYROLL_ACNT_NMBR", "CRT_DTTS", "SLRY_TRNS_DT", "FND_DT", "TOT_AMT",
                      "NMBR_OF_EMP", "FILE_TYP_EN_NM", "FILE_SUB_TYP_EN_NM", "STA_CD", "DLT_FLG"],
    "financing_schedules": ["loan_no", "client_no", "product", "account_debited", "contract_start_date",
                            "maturity_date", "instalment_no", "due_date", "principal", "profit", "instalment_amount",
                            "paid_date", "paid_amount", "status", "early_settlement_flag", "credit_limit_total",
                            "credit_limit_used", "credit_limit_available"],
    "deposits": ["deposit_id", "client_no", "account_no", "product", "amount", "start_date", "maturity_date",
                 "expected_profit", "rollover_instruction", "status"],
    "sadad_payments": ["payment_id", "account_no", "client_no", "biller_id", "biller_name", "biller_category",
                       "bill_number", "bill_issue_date", "amount", "due_date", "paid_date", "channel"],
    "code_lists": ["code_type", "code", "meaning"],
    "trade_finance": ["instrument_id", "client_no", "instrument_type", "subtype", "amount", "currency", "issue_date",
                      "expiry_date", "payment_date", "cash_margin_account", "cash_margin_amount", "claim_amount",
                      "claim_date", "status"],
    "card_sales": ["merchant_id", "client_no", "sales_date", "settlement_date", "sales_amount", "fees", "chargebacks",
                   "net_amount_paid"],
    "corporate_cards": ["record_id", "record_type", "client_no", "statement_date", "amount_due", "due_date",
                        "paid_date", "paid_amount", "topup_date", "topup_amount"],
    "bills_issued": ["invoice_id", "client_no", "customer_iban_code", "issue_date", "due_date", "amount", "paid_date",
                     "paid_amount"],
    "client_monthly_summary": ["client_no", "month", "txn_count_in", "txn_amount_in", "txn_count_out",
                               "txn_amount_out", "online_amount", "cheque_count", "avg_balance", "payroll_salary",
                               "payroll_bonus"],
    "group_structure": ["parent_client_no", "child_client_no", "relationship_type", "start_date"],
    "cheques": ["cheque_id", "account_no", "direction", "status", "received_date", "cheque_date", "cleared_date",
                "amount", "post_dated_flag"],
}

ABBREVIATIONS = {
    "client": "CSTMR", "customer": "CSTMR", "account": "ACNT", "no": "NMBR", "number": "NMBR", "date": "DT",
    "amount": "AMT", "txn": "TRNS", "code": "CD", "flag": "FLG", "type": "TYP", "name": "NM", "status": "STA_CD",
    "currency": "CCY", "balance": "BAL", "count": "CNT", "instruction": "INSTR", "beneficiary": "BNFCRY",
    "frequency": "FREQ", "created": "CRT", "changed": "CHG", "start": "STRT", "next": "NXT", "payment": "PYMT",
    "maturity": "MTRTY", "profit": "PRFT", "principal": "PRNCPL", "instalment": "INSTL", "expected": "EXPCTD",
    "rollover": "RLOVR", "deposit": "DPST", "product": "PROD", "issue": "ISSU", "expiry": "EXPRY",
    "settlement": "STLMNT", "sales": "SLS", "fees": "FEE", "chargebacks": "CHRGBCK", "statement": "STMT",
    "record": "REC", "merchant": "MRCHNT", "invoice": "INV", "bank": "BNK", "counterparty": "CNTRPRTY",
    "origin": "ORGN", "ref": "REF", "reversal": "RVRSL", "original": "ORGNL", "channel": "CHNL", "value": "VAL",
    "posting": "PSTNG", "exchange": "EXCH", "rate": "RT", "debit": "DBT", "credit": "CRDT", "ledger": "LDGR",
    "available": "AVLBL", "blocked": "BLCKD", "hold": "HLD", "overdraft": "OVRDFT", "limit": "LMT",
    "segment": "SGMNT", "sector": "SCTR", "industry": "INDSTRY", "region": "RGN", "joining": "JOIN", "legal": "LGL",
    "proprietor": "PRPRTR", "last": "LST", "annual": "ANNL", "employees": "EMP", "closure": "CLSR", "close": "CLS",
    "open": "OPN", "restricted": "RSTRCTD", "biller": "BLLR", "category": "CTGRY", "bill": "BL", "cheque": "CHQ",
    "direction": "DRCTN", "received": "RCVD", "cleared": "CLRD", "dated": "DTD", "instrument": "INSTRMNT",
    "subtype": "SUB_TYP", "cash": "CSH", "margin": "MRGN", "claim": "CLM", "contract": "CNTRCT", "debited": "DBTD",
    "early": "ERLY", "total": "TOT", "relationship": "RLTNSHP", "month": "MNTH", "avg": "AVG", "online": "ONLN",
    "meaning": "MNNG", "parent": "PRNT", "child": "CHLD", "party": "PRTY", "sub": "SUB", "party_type": "PRTY_TYP",
    "topup": "TOPUP", "due": "DUE", "paid": "PAID", "used": "USED", "own": "OWN", "tag": "TAG", "orig": "ORIG",
    "salary": "SLRY", "bonus": "BONUS", "loan": "LOAN", "net": "NET", "current": "CRNT", "accounts": "ACNTS",
    "postings": "PSTNG_CNT", "known": "KNWN",
}


def warehouse_column(column):
    """EDW-style name of a standard column: client_no -> CSTMR_CIF, value_date -> VAL_DT, own_account_flag -> OWN_ACNT_FLG."""
    if column.isupper():                        # already a warehouse name (payroll files)
        return column
    words = column.replace("client_no", "cstmr cif").replace("_", " ").split()
    return "_".join(w.upper() if w in ("cstmr", "cif") else ABBREVIATIONS.get(w, w.upper()) for w in words)


def is_flag(column):
    """Yes / no columns: true / false in the project, Y / N in the warehouse."""
    return column.endswith("_flag")


def staging_sql(table, schema):
    """SELECT that reads a warehouse table and gives back the standard column names and types."""
    warehouse_table = TABLES[table][0]
    select = []
    for column in COLUMNS[table]:
        source = f'"{warehouse_column(column)}"'
        select.append(f"({source} = 'Y') AS {column}" if is_flag(column) else f'{source} AS "{column}"')
    return (f"-- {warehouse_table} -> {table}: warehouse column names back to the standard names\n"
            "SELECT\n    " + ",\n    ".join(select) + f'\nFROM "{schema}"."{warehouse_table}"')


for _table, _columns in list(COLUMNS.items()) + list(DETAIL_COLUMNS.items()):   # names unique within a table
    _names = [warehouse_column(c) for c in _columns]
    assert len(set(_names)) == len(_names), (_table, _names)
'''

In [ ]:
LIBRARY_CODE = r'''
"""Shared settings and helpers for the cash-flow forecast recipes (project library: python/cashflow.py)."""
import numpy as np
import pandas as pd

# category code: (name, direction, predicted by the models?)
# Financing instalments and deposit maturities are fully scheduled: no model, their obligations are added as they are.
CATEGORIES = {
    "PAYROLL": ("Payroll", "OUT", True),
    "RENT": ("Rent", "OUT", True),
    "FINANCING": ("Financing instalments", "OUT", False),
    "SADAD_GOV": ("SADAD and government bills", "OUT", True),
    "SUPPLIERS": ("Supplier payments", "OUT", True),
    "CORP_CARD": ("Corporate card", "OUT", True),
    "OTHER_OUT": ("Other out", "OUT", True),
    "RECEIPTS": ("Customer receipts", "IN", True),
    "CARD_SALES": ("Card sales settlements", "IN", True),
    "CHEQUES_IN": ("Cheques in", "IN", True),
    "DEPOSIT_MAT": ("Deposit maturities", "IN", False),
    "OTHER_IN": ("Other in", "IN", True),
    "NET": ("Net cash flow", "NET", True),      # client total: money in minus money out
}
MODELLED = [c for c, (_, _, m) in CATEGORIES.items() if m]
MIN_HISTORY_MONTHS = 12     # a client needs this many months before the cut-off to get a forecast
HORIZONS = (1, 2, 3)        # months ahead
HOLDOUT_MONTHS = 6          # the last months kept aside to test the models
QUANTILES = (0.1, 0.9)      # P10 and P90
CATBOOST = {"iterations": 600, "depth": 6, "learning_rate": 0.08, "random_seed": 0, "verbose": 0}
FX_TO_SAR = {"SAR": 1.0, "USD": 3.75, "EUR": 4.05}

# Category rules: (origin type, transaction type tag, direction) -> category, the first matching rule wins;
# "*" matches anything. The warehouse gives money in and out by origin type and tag; these rules turn them into
# categories, the same way for postings and for scheduled payments. The setup copies them into the editable
# dataset category_rules: edit them there (Flow > category_rules > Edit), then run the scenario Monthly forecast.
DEFAULT_CATEGORY_RULES = [
    ("PAYROLL", "*", "*", "PAYROLL"),
    ("LOAN", "*", "OUT", "FINANCING"),
    ("SADAD", "*", "*", "SADAD_GOV"),
    ("CARD_SETTLEMENT", "*", "*", "CARD_SALES"),
    ("CORP_CARD", "*", "*", "CORP_CARD"),
    ("LC", "*", "*", "SUPPLIERS"),
    ("DEPOSIT", "*", "IN", "DEPOSIT_MAT"),
    ("CHEQUE", "*", "IN", "CHEQUES_IN"),
    ("INVOICE", "*", "*", "RECEIPTS"),
    ("*", "RENT", "*", "RENT"),
    ("*", "SUPPLIER", "OUT", "SUPPLIERS"),
    ("*", "COLLECTION", "IN", "RECEIPTS"),
    ("*", "SALARY", "*", "PAYROLL"),
    ("*", "UTILITY", "*", "SADAD_GOV"),
    ("*", "GOV", "*", "SADAD_GOV"),
    ("*", "*", "IN", "OTHER_IN"),
    ("*", "*", "OUT", "OTHER_OUT"),
]

# Amounts are divided by a usual size, so small and large clients and categories compare:
# a category's own amounts by its usual monthly amount (series_scale), the client's totals by its usual
# monthly flow (scale). The models predict the target divided by series_scale.
CAT_FEATURES = ["category", "direction", "segment", "sector", "region"]
SERIES_FEATURES = ["last_amount", "avg_3m", "same_month_last_year", "free_last", "free_avg_3m", "oblig_target_month"]
CLIENT_FEATURES = ["client_in_avg_3m", "client_out_avg_3m", "balance_end", "oblig_in_next_3m", "oblig_out_next_3m"]
OTHER_FEATURES = ["months_ahead", "target_month_of_year", "history_months", "log_scale", "series_share",
                  "log_annual_sales", "employees"]
FEATURES = CAT_FEATURES + SERIES_FEATURES + CLIENT_FEATURES + OTHER_FEATURES


def read_dataset(name):
    """A dataset as a DataFrame, with dates as plain (time zone free) timestamps."""
    import dataiku
    df = dataiku.Dataset(name).get_dataframe()
    for c in df.columns:
        if isinstance(df[c].dtype, pd.DatetimeTZDtype):
            df[c] = df[c].dt.tz_convert(None)
    return df


def month_start(dates):
    """First day of the month, as a plain (time zone free) timestamp."""
    d = pd.to_datetime(dates)
    if getattr(d.dt, "tz", None) is not None:
        d = d.dt.tz_convert(None)
    return d.dt.to_period("M").dt.to_timestamp()


def months_between(start, end):
    """Whole months from start to end (both month timestamps)."""
    return (end.dt.year - start.dt.year) * 12 + (end.dt.month - start.dt.month)


def add_months(month, n):
    return month + pd.offsets.DateOffset(months=n)


def cutoff_month():
    """The forecast's cut-off: the last complete month of the latest checked warehouse load (dataset load_status).
    Recipes that use it take load_status as an input, so nothing runs on a load that failed its checks."""
    status = read_dataset("load_status")
    return month_start(status["cutoff_month"]).iloc[0]


def apply_category_rules(df, rules):
    """Category of each row from its origin_type, txn_type_tag and direction: the first matching rule wins."""
    rules = rules.sort_values("rule_order")
    keys = {c: df[c].fillna("").astype(str) for c in ("origin_type", "txn_type_tag", "direction")}
    category = pd.Series(np.where(df["direction"] == "IN", "OTHER_IN", "OTHER_OUT"), index=df.index)
    unset = pd.Series(True, index=df.index)
    for r in rules.itertuples():
        match = unset.copy()
        for column in ("origin_type", "txn_type_tag", "direction"):
            wanted = getattr(r, column)
            if wanted != "*":
                match &= keys[column] == ("" if pd.isna(wanted) else str(wanted))
        category[match] = r.category
        unset &= ~match
    return category


def model_matrix(rows):
    rows = rows.assign(series_share=rows["series_scale"] / rows["scale"])
    X = rows[FEATURES].copy()
    for c in SERIES_FEATURES:
        X[c] = rows[c] / rows["series_scale"]
    for c in CLIENT_FEATURES:
        X[c] = rows[c] / rows["scale"]
    for c in CAT_FEATURES:
        X[c] = X[c].fillna("NA").astype(str)
    return X


def fit_models(rows):
    """EV model (expected value) and P10/P90 model (both quantiles in one model), on amounts / scale."""
    from catboost import CatBoostRegressor
    X, y = model_matrix(rows), rows["target"] / rows["series_scale"]
    ev = CatBoostRegressor(loss_function="RMSE", **CATBOOST)
    ev.fit(X, y, cat_features=CAT_FEATURES)
    alphas = ",".join(str(q) for q in QUANTILES)
    pq = CatBoostRegressor(loss_function=f"MultiQuantile:alpha={alphas}", **CATBOOST)
    pq.fit(X, y, cat_features=CAT_FEATURES)
    return {"ev": ev, "p10_p90": pq}


def finish(ev, p10, p90, category, index=None):
    """P10 <= EV <= P90, and no negative amounts except for the net cash flow (in SAR)."""
    ev, p10, p90 = (np.asarray(v, dtype=float) for v in (ev, p10, p90))
    p10, p90 = np.minimum(p10, ev), np.maximum(p90, ev)
    floor = np.where(np.asarray(category) == "NET", -np.inf, 0.0)
    return pd.DataFrame({"p10": np.maximum(p10, floor), "ev": np.maximum(ev, floor),
                         "p90": np.maximum(p90, floor)}, index=index).round(2)


def predict(models, rows):
    """P10, EV and P90 in SAR from the code models."""
    X = model_matrix(rows)
    scale = rows["series_scale"].values
    q = models["p10_p90"].predict(X)
    return finish(models["ev"].predict(X) * scale, q[:, 0] * scale, q[:, 1] * scale, rows["category"], rows.index)
'''

In [ ]:
RECIPES, SQL_RECIPES = {}, {}

In [ ]:
RECIPES["load_check"] = r'''
# Load check: the latest warehouse load must be complete before anything else runs.
# Reads CFF_LOAD_RECORD; for the latest load, every table must be loaded with status SUCCESS, and each table must
# hold the number of rows the load record says (a partial load stops here). Writes load_status: the load's
# as-of date and the forecast's cut-off (the last complete month on that date). On any problem the recipe fails,
# so nothing downstream runs.
import dataiku
import pandas as pd
from dataiku import SQLExecutor2

from cashflow import read_dataset
from warehouse import LOAD_RECORD, TABLES

record = read_dataset(LOAD_RECORD)
latest = record.sort_values("LOAD_DTTS").iloc[-1]["LOAD_ID"]
load = record[record["LOAD_ID"] == latest].set_index("TRGT_TBL_NM")
problems = []
for table, (warehouse_table, _) in TABLES.items():
    if warehouse_table not in load.index:
        problems.append(f"{warehouse_table}: not in load {latest}")
        continue
    row = load.loc[warehouse_table]
    if row["LOAD_STA_CD"] != "SUCCESS":
        problems.append(f"{warehouse_table}: load status {row['LOAD_STA_CD']}")
    params = dataiku.Dataset(warehouse_table).get_config()["params"]
    count = SQLExecutor2(dataset=dataiku.Dataset(warehouse_table)).query_to_df(
        f'SELECT COUNT(*) AS n FROM "{params["schema"]}"."{params["table"]}"')["n"].iloc[0]
    if count != row["ROW_CNT"]:
        problems.append(f"{warehouse_table}: {count:,} rows, the load record says {row['ROW_CNT']:,}")
if problems:
    raise ValueError("Warehouse load not usable:\n" + "\n".join(problems))

as_of = pd.Timestamp(load["AS_OF_DT"].max()).normalize()
month_end = as_of + pd.offsets.MonthEnd(0)
cutoff = as_of.to_period("M") if as_of == month_end else as_of.to_period("M") - 1
dataiku.Dataset("load_status").write_with_schema(pd.DataFrame([{
    "load_id": latest, "loaded_at": load["LOAD_DTTS"].max(), "as_of_date": as_of,
    "cutoff_month": cutoff.to_timestamp(), "tables_checked": len(TABLES), "checked_at": pd.Timestamp.now().floor("s")}]))
print(f"Load {latest} OK: as of {as_of:%Y-%m-%d}, cut-off month {cutoff}")
'''

In [ ]:
RECIPES["categorise"] = r'''
# Monthly flows: money in and out per client, month and category. The warehouse gives them by origin type and
# transaction type tag (own-account transfers and reversals already left out); the editable category_rules turn
# those into categories.
import dataiku

from cashflow import apply_category_rules, month_start, read_dataset

flows = read_dataset("monthly_flows_by_origin")
flows["month"] = month_start(flows["month"])
flows["category"] = apply_category_rules(flows, read_dataset("category_rules"))
monthly = (flows.groupby(["client_no", "month", "category", "direction"], as_index=False)
           [["amount", "known_1", "known_2", "known_3", "postings"]].sum())
dataiku.Dataset("monthly_flows").write_with_schema(monthly.round(2))
'''

In [ ]:
RECIPES["obligations"] = r'''
# Obligations: the scheduled payments (from the warehouse, past and future) per client, category, month due and
# month known. Categories come from the same category_rules as the monthly flows.
import dataiku
import pandas as pd

from cashflow import add_months, apply_category_rules, cutoff_month, month_start, read_dataset

horizon_end = add_months(cutoff_month(), 4)     # needed up to 3 months after the cut-off
items = read_dataset("scheduled_payments")
items = items[pd.to_datetime(items["due_date"]) < horizon_end]
items["category"] = apply_category_rules(items, read_dataset("category_rules"))
items["due_month"] = month_start(items["due_date"])
items["known_month"] = month_start(items["known_date"])
monthly_obligations = (items.groupby(["client_no", "category", "direction", "due_month", "known_month"], as_index=False)
                       .agg(amount=("amount", "sum"), items=("origin_ref", "count")))
dataiku.Dataset("monthly_obligations").write_with_schema(monthly_obligations)
'''

In [ ]:
RECIPES["build_rows"] = r'''
# Build rows: one row per client, category, cut-off month and months ahead (1, 2, 3).
# Inputs only use what was known at the end of the cut-off month; the answer (target) is the amount in the
# predicted month minus the obligations already known at the cut-off.
# training_rows: every past cut-off with an answer. scoring_rows: the latest cut-off, to forecast.
import dataiku
import numpy as np
import pandas as pd

from cashflow import (CATEGORIES, HORIZONS, MIN_HISTORY_MONTHS, MODELLED, add_months, cutoff_month, month_start,
                      months_between, read_dataset)

last_month = cutoff_month()
flows = read_dataset("monthly_flows")
flows["month"] = month_start(flows["month"])
obl = read_dataset("monthly_obligations")
obl["due_month"], obl["known_month"] = month_start(obl["due_month"]), month_start(obl["known_month"])
clients = read_dataset("clients")

# ---- every client's months, every category, zeros where nothing moved ----------------------------
span = flows.groupby("client_no")["month"].agg(first_month="min", last_seen="max").reset_index()
span = span.merge(clients[["client_no", "status"]], on="client_no")
span["end_month"] = np.where(span["status"] == "ACTIVE", last_month, span["last_seen"])
real = [c for c in CATEGORIES if c != "NET"]
grid = pd.DataFrame([(r.client_no, m, c) for r in span.itertuples()
                     for m in pd.date_range(r.first_month, r.end_month, freq="MS") for c in real],
                    columns=["client_no", "month", "category"])
grid["direction"] = grid["category"].map({c: v[1] for c, v in CATEGORIES.items()})
values = ["amount", "known_1", "known_2", "known_3"]
grid = grid.merge(flows[["client_no", "month", "category"] + values], on=["client_no", "month", "category"], how="left")
grid[values] = grid[values].fillna(0.0)

# client totals per month, and the net cash flow as one more series
sign = np.where(grid["direction"] == "IN", 1.0, -1.0)
totals = (grid.assign(money_in=grid["amount"].where(sign > 0, 0.0), money_out=grid["amount"].where(sign < 0, 0.0))
          .groupby(["client_no", "month"], as_index=False)[["money_in", "money_out"]].sum())
net = (grid[values].mul(sign, axis=0).assign(client_no=grid["client_no"], month=grid["month"])
       .groupby(["client_no", "month"], as_index=False)[values].sum().assign(category="NET", direction="NET"))
series = pd.concat([grid, net], ignore_index=True).sort_values(["client_no", "category", "month"])

# ---- inputs at each cut-off (the month itself is the cut-off) -----------------------------------
g = series.groupby(["client_no", "category"])
roll = lambda col, n: g[col].transform(lambda s: s.rolling(n, min_periods=1).mean())
series["free"] = series["amount"] - series["known_1"]
series["last_amount"] = series["amount"]
series["avg_3m"], series["avg_12m"] = roll("amount", 3), roll("amount", 12)
series["free_last"] = series["free"]
series["free_avg_3m"] = series.groupby(["client_no", "category"])["free"].transform(
    lambda s: s.rolling(3, min_periods=1).mean())
for k in (9, 10, 11):
    series[f"lag_{k}"] = g["amount"].shift(k)

totals = totals.sort_values(["client_no", "month"])
t = totals.groupby("client_no")
for col, short in (("money_in", "in"), ("money_out", "out")):
    totals[f"client_{short}_avg_3m"] = t[col].transform(lambda s: s.rolling(3, min_periods=1).mean())
    totals[f"client_{short}_avg_12m"] = t[col].transform(lambda s: s.rolling(12, min_periods=1).mean())
totals["scale"] = ((totals["client_in_avg_12m"] + totals["client_out_avg_12m"]) / 2).clip(lower=1000.0)

bal = read_dataset("monthly_balances")
bal["month"] = month_start(bal["month"])

base = (series.merge(totals[["client_no", "month", "client_in_avg_3m", "client_out_avg_3m", "scale"]],
                     on=["client_no", "month"], how="left")
        .merge(bal, on=["client_no", "month"], how="left")
        .merge(span[["client_no", "first_month", "end_month"]], on="client_no"))
base["balance_end"] = base["balance_end"].fillna(0.0)
base["history_months"] = months_between(base["first_month"], base["month"]) + 1

# ---- obligations known at the cut-off ----------------------------------------------------------
expanded = []
for h in HORIZONS:
    o = obl.assign(cutoff=obl["due_month"].apply(lambda m: add_months(m, -h)), months_ahead=h)
    expanded.append(o[o["known_month"] <= o["cutoff"]])
expanded = pd.concat(expanded, ignore_index=True)
ob_cat = expanded.groupby(["client_no", "category", "cutoff", "months_ahead"], as_index=False)["amount"].sum()
signed = expanded.assign(amount=np.where(expanded["direction"] == "IN", 1.0, -1.0) * expanded["amount"])
ob_net = signed.groupby(["client_no", "cutoff", "months_ahead"], as_index=False)["amount"].sum().assign(category="NET")
ob_target = pd.concat([ob_cat, ob_net], ignore_index=True).rename(columns={"amount": "oblig_target_month"})
ob_next = (expanded.pivot_table(index=["client_no", "cutoff"], columns="direction", values="amount", aggfunc="sum",
                                fill_value=0.0)
           .rename(columns={"IN": "oblig_in_next_3m", "OUT": "oblig_out_next_3m"}).reset_index())
for c in ("oblig_in_next_3m", "oblig_out_next_3m"):
    if c not in ob_next:
        ob_next[c] = 0.0

# ---- one row per months ahead, with its answer -------------------------------------------------
answers = series[["client_no", "category", "month", "amount", "known_1", "known_2", "known_3"]].rename(
    columns={"month": "target_month", "amount": "actual"})
profile = clients[["client_no", "segment", "sector", "region", "last_annual_sales", "number_of_employees"]]
rows = []
for h in HORIZONS:
    r = base.assign(months_ahead=h, target_month=base["month"].apply(lambda m: add_months(m, h)))
    r["same_month_last_year"] = r[f"lag_{12 - h}"]
    r = r.merge(answers, on=["client_no", "category", "target_month"], how="left")
    r["obligations_known"] = r[f"known_{h}_y"]
    r["target"] = r["actual"] - r["obligations_known"]
    rows.append(r)
rows = pd.concat(rows, ignore_index=True).rename(columns={"month": "cutoff_month"})
rows = rows[rows["category"].isin(MODELLED)]
rows = (rows.merge(ob_target.rename(columns={"cutoff": "cutoff_month"}),
                   on=["client_no", "category", "cutoff_month", "months_ahead"], how="left")
        .merge(ob_next.rename(columns={"cutoff": "cutoff_month"}), on=["client_no", "cutoff_month"], how="left")
        .merge(profile, on="client_no", how="left"))
fill = ["oblig_target_month", "oblig_in_next_3m", "oblig_out_next_3m", "same_month_last_year"]
rows[fill] = rows[fill].fillna(0.0)
rows["target_month_of_year"] = rows["target_month"].dt.month
rows["log_scale"] = np.log10(rows["scale"])
rows["log_annual_sales"] = np.log10(rows["last_annual_sales"].clip(lower=1.0))
rows["employees"] = rows["number_of_employees"]
rows["enough_history"] = rows["history_months"] >= MIN_HISTORY_MONTHS
# a category the client did not use in the last 12 months is forecast as 0 (plus its obligations), not modelled
rows["active"] = (rows["category"] == "NET") | (rows["avg_12m"] > 0)
rows["series_scale"] = np.where(rows["category"] == "NET", rows["scale"],
                                np.maximum(rows["avg_12m"], 0.01 * rows["scale"]))

keep = ["client_no", "category", "direction", "cutoff_month", "months_ahead", "target_month", "history_months",
        "enough_history", "active", "scale", "series_scale", "segment", "sector", "region", "last_amount", "avg_3m",
        "avg_12m", "same_month_last_year", "free_last", "free_avg_3m", "client_in_avg_3m", "client_out_avg_3m",
        "balance_end", "oblig_target_month", "oblig_in_next_3m", "oblig_out_next_3m", "target_month_of_year",
        "log_scale", "log_annual_sales", "employees", "actual", "obligations_known", "target"]
rows = rows[keep]
training = rows[rows["enough_history"] & rows["active"] & rows["target"].notna()]
scoring = rows[(rows["cutoff_month"] == last_month)].drop(columns=["actual", "obligations_known", "target"])
dataiku.Dataset("training_rows").write_with_schema(training.round(4).reset_index(drop=True))
dataiku.Dataset("scoring_rows").write_with_schema(scoring.round(4).reset_index(drop=True))
'''

In [ ]:
RECIPES["train_models"] = r'''
# Train the two models (EV, and P10/P90 together) twice:
#   backtest: only rows whose predicted month is before the last 6 months (used by the test step)
#   final:    all rows (used to forecast the next 3 months)
import json
import pickle

import dataiku
import pandas as pd

from cashflow import (CATBOOST, FEATURES, HOLDOUT_MONTHS, QUANTILES, add_months, cutoff_month, fit_models,
                      month_start, read_dataset)

last_month = cutoff_month()
holdout_start = add_months(last_month, -(HOLDOUT_MONTHS - 1))
rows = read_dataset("training_rows")
rows["target_month"] = month_start(rows["target_month"])
folder = dataiku.Folder("models")

info = {"last_month": str(last_month.date()), "holdout_start": str(holdout_start.date()), "features": FEATURES,
        "quantiles": list(QUANTILES), "catboost": CATBOOST}
for name, part in (("backtest", rows[rows["target_month"] < holdout_start]), ("final", rows)):
    models = fit_models(part)
    for kind, model in models.items():
        with folder.get_writer(f"{name}_{kind}.pkl") as w:
            pickle.dump(model, w)
    importance = models["ev"].get_feature_importance(prettified=True)
    info[name] = {"rows": len(part), "clients": int(part["client_no"].nunique()),
                  "target_months": [str(part["target_month"].min().date()), str(part["target_month"].max().date())],
                  "ev_feature_importance": dict(zip(importance["Feature Id"], importance["Importances"].round(2)))}
    print(name, info[name]["rows"], "rows")
with folder.get_writer("info.json") as w:
    w.write(json.dumps(info, indent=2).encode())
'''

In [ ]:
RECIPES["test_models"] = r'''
# Test: score the last 6 months with the backtest models (trained without them) and compare with what happened.
#   EV error (WAPE) = sum |actual - EV| / sum |actual|, next to a simple guess (average of the last 3 months)
#   coverage = share of actual months between P10 and P90 (aim: about 80%)
import pickle

import dataiku
import numpy as np
import pandas as pd

from cashflow import CATEGORIES, HOLDOUT_MONTHS, add_months, cutoff_month, month_start, predict, read_dataset

last_month = cutoff_month()
holdout_start = add_months(last_month, -(HOLDOUT_MONTHS - 1))
rows = read_dataset("training_rows")
rows["target_month"] = month_start(rows["target_month"])
test = rows[rows["target_month"] >= holdout_start].copy()

folder = dataiku.Folder("models")
models = {}
for kind in ("ev", "p10_p90"):
    with folder.get_download_stream(f"backtest_{kind}.pkl") as s:
        models[kind] = pickle.loads(s.read())

test = test.join(predict(models, test))
test["baseline"] = np.where(test["category"] == "NET", test["free_avg_3m"], test["free_avg_3m"].clip(lower=0))
ids = ["client_no", "category", "direction", "cutoff_month", "months_ahead", "target_month", "actual",
       "obligations_known", "target", "p10", "ev", "p90", "baseline"]
dataiku.Dataset("test_predictions").write_with_schema(test[ids].reset_index(drop=True))


def scores(d):
    size = d["target"].abs().sum()
    inside = (d["target"] >= d["p10"]) & (d["target"] <= d["p90"])
    return pd.Series({"rows": len(d), "actual_total": round(d["target"].sum(), 0),
                      "wape_model": round((d["target"] - d["ev"]).abs().sum() / size, 4) if size else np.nan,
                      "wape_baseline": round((d["target"] - d["baseline"]).abs().sum() / size, 4) if size else np.nan,
                      "coverage_p10_p90": round(inside.mean(), 4),
                      "below_p10": round((d["target"] < d["p10"]).mean(), 4),
                      "above_p90": round((d["target"] > d["p90"]).mean(), 4)})


results = [test.groupby(["category", "months_ahead"]).apply(scores).reset_index(),
           test.groupby("category").apply(scores).reset_index().assign(months_ahead="ALL")]
categories_only = test[test["category"] != "NET"]
results.append(categories_only.groupby("months_ahead").apply(scores).reset_index().assign(category="ALL"))
results.append(scores(categories_only).to_frame().T.assign(category="ALL", months_ahead="ALL"))
results = pd.concat(results, ignore_index=True)
results["months_ahead"] = results["months_ahead"].astype(str)
results["category_name"] = results["category"].map({c: v[0] for c, v in CATEGORIES.items()}).fillna("All categories")
order = ["category", "category_name", "months_ahead", "rows", "actual_total", "wape_model", "wape_baseline",
         "coverage_p10_p90", "below_p10", "above_p90"]
results = results[order].sort_values(["category", "months_ahead"]).reset_index(drop=True)
results["test_months"] = f"{holdout_start:%Y-%m} to {last_month:%Y-%m}"
dataiku.Dataset("test_results").write_with_schema(results)
'''

In [ ]:
RECIPES["ml_rows"] = r'''
# Model-ready rows for the visual models: the same inputs the code models see (amounts divided by their usual
# size), and the answer divided the same way (target_scaled). ml_train: rows whose predicted month is before the
# last 6 months; ml_test: the last 6 months; ml_scoring: the latest cut-off, for clients and categories to forecast.
import dataiku
import pandas as pd

from cashflow import HOLDOUT_MONTHS, add_months, cutoff_month, model_matrix, month_start, read_dataset

last_month = cutoff_month()
holdout_start = add_months(last_month, -(HOLDOUT_MONTHS - 1))


def model_ready(rows, with_target=True):
    out = rows[["client_no", "cutoff_month", "target_month", "series_scale"]].join(model_matrix(rows))
    if with_target:
        out["target"] = rows["target"]
        out["target_scaled"] = rows["target"] / rows["series_scale"]
    return out.reset_index(drop=True)


training = read_dataset("training_rows")
training["target_month"] = month_start(training["target_month"])
scoring = read_dataset("scoring_rows")
scoring = scoring[scoring["enough_history"].astype(bool) & scoring["active"].astype(bool)]

dataiku.Dataset("ml_train").write_with_schema(model_ready(training[training["target_month"] < holdout_start]))
dataiku.Dataset("ml_test").write_with_schema(model_ready(training[training["target_month"] >= holdout_start]))
dataiku.Dataset("ml_scoring").write_with_schema(model_ready(scoring, with_target=False))
'''

In [ ]:
RECIPES["score"] = r'''
# Score: P10, EV and P90 of the part the models predict, for each of the next 3 months (final models).
# Clients with less than 12 months of history get no prediction; a category the client did not use in the
# last 12 months gets 0.
import pickle

import dataiku
import pandas as pd

from cashflow import predict, read_dataset

rows = read_dataset("scoring_rows")
folder = dataiku.Folder("models")
models = {}
for kind in ("ev", "p10_p90"):
    with folder.get_download_stream(f"final_{kind}.pkl") as s:
        models[kind] = pickle.loads(s.read())

ok = rows["enough_history"].astype(bool) & rows["active"].astype(bool)
rows = rows.join(predict(models, rows[ok]))
unused = rows["enough_history"].astype(bool) & ~rows["active"].astype(bool)
rows.loc[unused, ["p10", "ev", "p90"]] = 0.0
cols = ["client_no", "category", "direction", "cutoff_month", "months_ahead", "target_month", "history_months",
        "enough_history", "p10", "ev", "p90"]
dataiku.Dataset("predictions").write_with_schema(rows[cols])
'''

In [ ]:
RECIPES["visual_predictions"] = r'''
# Visual predictions: the three visual models' predictions (EV, P10, P90) back in SAR, in the same shape as the
# code models' predictions.
import dataiku

from cashflow import finish, read_dataset

keys = ["client_no", "category", "cutoff_month", "months_ahead"]
rows = read_dataset("scored_ev_visual")[keys + ["direction", "target_month", "series_scale", "prediction"]]
rows = rows.rename(columns={"prediction": "ev"})
for q in ("p10", "p90"):
    scored = read_dataset(f"scored_{q}_visual")[keys + ["prediction"]].rename(columns={"prediction": q})
    rows = rows.merge(scored, on=keys)
scale = rows["series_scale"]
rows[["p10", "ev", "p90"]] = finish(rows["ev"] * scale, rows["p10"] * scale, rows["p90"] * scale, rows["category"],
                                    rows.index)
cols = ["client_no", "category", "direction", "cutoff_month", "months_ahead", "target_month", "p10", "ev", "p90"]
dataiku.Dataset("predictions_visual").write_with_schema(rows[cols])
'''

In [ ]:
RECIPES["compare_models"] = r'''
# Compare the two ways of training on the same last 6 months (both trained without them):
#   EV error (WAPE) = sum |actual - EV| / sum |actual|, and the share of actual months between P10 and P90.
# The project variable model_source picks which one feeds the final forecast.
import dataiku
import numpy as np
import pandas as pd

from cashflow import CATEGORIES, finish, read_dataset

keys = ["client_no", "category", "cutoff_month", "months_ahead"]
code = read_dataset("test_predictions")[keys + ["target", "p10", "ev", "p90"]].assign(source="code")

visual = read_dataset("eval_ev_visual_scored")[keys + ["series_scale", "target", "prediction"]]
visual = visual.rename(columns={"prediction": "ev"})
for q in ("p10", "p90"):
    scored = read_dataset(f"eval_{q}_visual_scored")[keys + ["prediction"]].rename(columns={"prediction": q})
    visual = visual.merge(scored, on=keys)
s = visual["series_scale"]
visual[["p10", "ev", "p90"]] = finish(visual["ev"] * s, visual["p10"] * s, visual["p90"] * s, visual["category"],
                                      visual.index)
both = pd.concat([code, visual.drop(columns="series_scale").assign(source="visual")], ignore_index=True)


def scores(d):
    size = d["target"].abs().sum()
    return pd.Series({"rows": len(d),
                      "wape": round((d["target"] - d["ev"]).abs().sum() / size, 4) if size else np.nan,
                      "coverage_p10_p90": round(((d["target"] >= d["p10"]) & (d["target"] <= d["p90"])).mean(), 4),
                      "below_p10": round((d["target"] < d["p10"]).mean(), 4),
                      "above_p90": round((d["target"] > d["p90"]).mean(), 4)})


per_category = both.groupby(["category", "source"]).apply(scores).reset_index()
overall = both[both["category"] != "NET"].groupby("source").apply(scores).reset_index().assign(category="ALL")
table = pd.concat([overall, per_category], ignore_index=True)
table["category_name"] = table["category"].map({c: v[0] for c, v in CATEGORIES.items()}).fillna("All categories")
wide = table.pivot(index=["category", "category_name"], columns="source",
                   values=["wape", "coverage_p10_p90", "below_p10", "above_p90", "rows"])
wide.columns = [f"{metric}_{source}" for metric, source in wide.columns]
wide = wide.reset_index()
wide["better_ev"] = np.where(wide["wape_code"] <= wide["wape_visual"], "code", "visual")
dataiku.Dataset("model_comparison").write_with_schema(wide)
'''

In [ ]:
RECIPES["final_forecast"] = r'''
# Final forecast: the models' prediction plus the obligations due that month (already scheduled, so the same
# amount is added to P10, EV and P90). Financing instalments and deposit maturities are obligations only.
# "NET" is the client total (money in minus money out) with its own P10, EV and P90.
# The project variable model_source picks the models: "code" (CatBoost recipes) or "visual" (visual ML).
import dataiku
import numpy as np
import pandas as pd

from cashflow import CATEGORIES, HORIZONS, cutoff_month, month_start, read_dataset

last_month = cutoff_month()
source = dataiku.get_custom_variables().get("model_source", "code")
pred = read_dataset("predictions")
pred["target_month"] = month_start(pred["target_month"])
obl = read_dataset("monthly_obligations")
obl["due_month"], obl["known_month"] = month_start(obl["due_month"]), month_start(obl["known_month"])

clients = pred[["client_no", "history_months", "enough_history"]].drop_duplicates("client_no")
frame = clients.merge(pd.DataFrame([(c, h) for c in CATEGORIES for h in HORIZONS], columns=["category", "months_ahead"]),
                      how="cross")
frame["cutoff_month"] = last_month
frame["target_month"] = frame["months_ahead"].apply(lambda h: last_month + pd.offsets.DateOffset(months=h))
frame["direction"] = frame["category"].map({c: v[1] for c, v in CATEGORIES.items()})
frame["category_name"] = frame["category"].map({c: v[0] for c, v in CATEGORIES.items()})

known = obl[obl["known_month"] <= last_month]
by_cat = known.groupby(["client_no", "category", "due_month"], as_index=False)["amount"].sum()
net = (known.assign(amount=np.where(known["direction"] == "IN", 1.0, -1.0) * known["amount"])
       .groupby(["client_no", "due_month"], as_index=False)["amount"].sum().assign(category="NET"))
dues = pd.concat([by_cat, net], ignore_index=True).rename(columns={"due_month": "target_month", "amount": "obligations"})
frame = frame.merge(dues, on=["client_no", "category", "target_month"], how="left")
frame["obligations"] = frame["obligations"].fillna(0.0)

model = pred if source == "code" else read_dataset("predictions_visual")
model = model[["client_no", "category", "months_ahead", "p10", "ev", "p90"]].rename(
    columns={"p10": "model_p10", "ev": "model_ev", "p90": "model_p90"})
frame = frame.merge(model, on=["client_no", "category", "months_ahead"], how="left")
enough = frame["enough_history"].astype(bool)
for q in ("p10", "ev", "p90"):
    # no model value: a category fully scheduled, or one the client did not use in the last 12 months
    frame.loc[enough & frame[f"model_{q}"].isna(), f"model_{q}"] = 0.0
    frame[q] = frame[f"model_{q}"] + frame["obligations"]
frame.loc[~enough, ["p10", "ev", "p90", "model_p10", "model_ev", "model_p90"]] = np.nan
frame["status"] = np.where(enough, "OK", "NOT_ENOUGH_HISTORY")
frame["model_source"] = source

cols = ["client_no", "cutoff_month", "months_ahead", "target_month", "category", "category_name", "direction",
        "p10", "ev", "p90", "obligations", "model_p10", "model_ev", "model_p90", "history_months", "status",
        "model_source"]
order = {c: i for i, c in enumerate(CATEGORIES)}
frame = frame.sort_values(["client_no", "months_ahead"]).sort_values(
    ["client_no", "months_ahead", "category"], key=lambda s: s.map(order) if s.name == "category" else s)
dataiku.Dataset("final_forecast").write_with_schema(frame[cols].round(2).reset_index(drop=True))
'''

## 3. Build the flow

Links the curated warehouse tables (read-only), creates the editable category rules, the staging recipes, the
load check, the data prep and the code training. The code models' folder is only rebuilt when asked by name, like saved models, so building the
forecast never retrains by accident.

In [ ]:
import time

import pandas as pd

flow = project.get_flow()
if project.list_recipes():
    raise RuntimeError("This project already has a flow. This notebook sets up an empty project only; "
                       "change the flow in Dataiku, or create a new project to set it up again.")
lib = {}
exec(LIBRARY_CODE, lib)     # the shared settings (FEATURES, ...) are needed here too, for the visual models
exec(WAREHOUSE_CODE, lib)   # the curated warehouse tables (TABLES, COLUMNS, staging_sql)
RAW = [warehouse_table for warehouse_table, _ in lib["TABLES"].values()] + [lib["LOAD_RECORD"]]

# recipe: (inputs, outputs, flow zone); "models" is a managed folder, everything else a dataset
FLOW = {
    "load_check": (RAW, ["load_status"], "Staging"),
    "categorise": (["monthly_flows_by_origin", "category_rules"], ["monthly_flows"], "Data prep"),
    "obligations": (["load_status", "scheduled_payments", "category_rules"], ["monthly_obligations"], "Data prep"),
    "build_rows": (["load_status", "monthly_flows", "monthly_obligations", "clients", "monthly_balances"],
                   ["training_rows", "scoring_rows"], "Data prep"),
    "train_models": (["load_status", "training_rows"], ["models"], "Training (code)"),
    "test_models": (["load_status", "training_rows", "models"], ["test_predictions", "test_results"],
                    "Training (code)"),
    "ml_rows": (["load_status", "training_rows", "scoring_rows"], ["ml_train", "ml_test", "ml_scoring"],
                "Training (visual)"),
    "score": (["scoring_rows", "models"], ["predictions"], "Inference"),
}
# created after the visual models exist (they read their Evaluate and Score outputs)
FLOW_AFTER_VISUAL = {
    "visual_predictions": (["scored_ev_visual", "scored_p10_visual", "scored_p90_visual"], ["predictions_visual"],
                           "Inference"),
    "compare_models": (["test_predictions", "eval_ev_visual_scored", "eval_p10_visual_scored",
                        "eval_p90_visual_scored"], ["model_comparison"], "Inference"),
    "final_forecast": (["load_status", "predictions", "predictions_visual", "monthly_obligations"], ["final_forecast"],
                       "Inference"),
}

variables = project.get_variables()
variables["standard"]["model_source"] = MODEL_SOURCE
project.set_variables(variables)

library = project.get_library()
python_folder = library.get_folder("python") or library.add_folder("python")
for file_name, code in (("cashflow.py", LIBRARY_CODE), ("warehouse.py", WAREHOUSE_CODE)):
    (python_folder.get_file(file_name) or python_folder.add_file(file_name)).write(code)


def new_dataset(name):
    """Managed dataset of the project on CONNECTION; on a SQL connection its table is <PROJECTKEY>_<name>."""
    dataset = project.new_managed_dataset(name).with_store_into(CONNECTION).create()
    settings = dataset.get_settings()
    if "table" in settings.get_raw()["params"]:
        settings.get_raw()["params"]["table"] = "${projectKey}_" + name
        settings.save()


# the curated warehouse tables: read-only, through the EDW_AI connection
for name in RAW:
    dataset = project.create_dataset(name, "PostgreSQL", params={
        "connection": EDW_CONNECTION, "mode": "table", "schema": EDW_SCHEMA, "table": name})
    dataset.autodetect_settings().save()
print(f"{len(RAW)} warehouse tables linked (read-only, {EDW_CONNECTION})")

# category rules: an editable dataset (Flow > category_rules > Edit), filled with the default rules
project.create_dataset("category_rules", "Inline", params={}, formatType="", formatParams={})
dataiku.Dataset("category_rules").write_with_schema(pd.DataFrame(
    [(n, *rule) for n, rule in enumerate(lib["DEFAULT_CATEGORY_RULES"], 1)],
    columns=["rule_order", "origin_type", "txn_type_tag", "direction", "category"]))

models = project.create_managed_folder("models", connection_name=FOLDER_CONNECTION)
folder_settings = models.get_settings()
folder_settings.get_raw()["flowOptions"]["rebuildBehavior"] = "EXPLICIT"   # retrained only when asked
folder_settings.save()
ref = lambda name: models.id if name == "models" else name
existing = {d["name"] for d in project.list_datasets()}


def recipe(kind, name, inputs, outputs, code):
    """A Python or SQL recipe (SQL runs inside the database)."""
    for out in outputs:
        if out != "models" and out not in existing:
            new_dataset(out)
            existing.add(out)
    builder = project.new_recipe("python" if kind == "python" else "sql_query", name)
    for i in inputs:
        builder.with_input(ref(i))
    for o in outputs:
        builder.with_output(ref(o))
    created = builder.create()
    settings = created.get_settings()
    settings.set_payload(code)
    if kind == "python":
        settings.set_code_env(code_env=CODE_ENV)
    settings.save()
    if kind == "sql" and all(i in RAW for i in inputs):    # other SQL recipes get their schema on their first run
        created.compute_schema_updates().apply()
    print(f"{kind} recipe", name, "->", ", ".join(outputs))


# staging: one SQL recipe per warehouse table, warehouse names back to the standard names
for table, (warehouse_table, _) in lib["TABLES"].items():
    recipe("sql", f"stage_{table}", [warehouse_table], [table], lib["staging_sql"](table, EDW_SCHEMA))
for name, (inputs, outputs, _) in FLOW.items():
    recipe("sql" if name in SQL_RECIPES else "python", name, inputs, outputs, SQL_RECIPES.get(name) or RECIPES[name])

## 4. Check the load and prepare the data

Runs the load check, the staging and the data prep. The visual models need their data to exist before they can
be trained.

In [ ]:
started = time.time()
job = project.new_job("RECURSIVE_FORCED_BUILD").with_auto_update_schema_before_each_recipe_run(True)
for name in ["ml_train", "ml_test", "ml_scoring"] + list(lib["TABLES"]):
    job.with_output(name)
job.start_and_wait()
status = dataiku.Dataset("load_status").get_dataframe()
print(f"Load {status['load_id'][0]} checked: as of {status['as_of_date'][0]:%Y-%m-%d}, "
      f"cut-off month {status['cutoff_month'][0]:%Y-%m}")
print(f"Staging and data prep built in {(time.time() - started) / 60:.1f} minutes")

## 5. Visual models and scenarios

Three visual ML tasks in the Lab (Lab > Visual analyses), trained on `ml_train` and tested on `ml_test`:

- **EV (visual)**: LightGBM and XGBoost, the better one (lowest RMSE) is deployed.
- **P10 (visual)** and **P90 (visual)**: LightGBM set to predict the 10th and 90th percentile (a custom
  algorithm in the Lab), scored with the pinball loss and the share of actual values below the prediction.

Each is deployed to the flow as a saved model, with an Evaluate recipe on the last 6 months (results kept in a
model evaluation store, so every retraining is tracked) and a Score recipe on the latest cut-off.

Then the two scenarios, with their monthly triggers switched off (switch them on in production):

- **Monthly forecast**: check the latest load, then stage, prepare and forecast with the current models.
- **Retrain and test the models**: retrain both ways, test them on the last 6 months, compare, forecast.

In [ ]:
from dataikuapi.dss.utils import DSSDatasetSelectionBuilder

QUANTILE_MODEL = """
from lightgbm import LGBMRegressor
clf = LGBMRegressor(objective="quantile", alpha={alpha}, n_estimators=500, learning_rate=0.05, num_leaves=31,
                    min_child_samples=50, verbose=-1)
"""
PINBALL = """
import numpy as np
def score(y_valid, y_pred, eval_df=None, output_df=None, ref_sample_df=None, **kwargs):
    diff = np.asarray(y_valid) - np.asarray(y_pred)
    return float(np.mean(np.maximum({alpha} * diff, ({alpha} - 1) * diff)))
"""
SHARE_BELOW = """
import numpy as np
def score(y_valid, y_pred, eval_df=None, output_df=None, ref_sample_df=None, **kwargs):
    return float(np.mean(np.asarray(y_valid) < np.asarray(y_pred)))
"""
# saved model: (Lab analysis name, quantile or None for the expected value)
VISUAL_MODELS = {"ev_visual": ("EV (visual)", None), "p10_visual": ("P10 (visual)", 0.1),
                 "p90_visual": ("P90 (visual)", 0.9)}
full = DSSDatasetSelectionBuilder().with_all_data_sampling().build()


def lab_model(analysis_name, alpha):
    """Visual ML task on ml_train (tested on ml_test), trained in the Lab; returns the task and its best model."""
    task = project.create_prediction_ml_task(input_dataset="ml_train", target_variable="target_scaled",
                                             prediction_type="REGRESSION", wait_guess_complete=True)
    definition = project.get_analysis(task.analysis_id).get_definition()
    definition.get_raw()["name"] = analysis_name
    definition.save()
    settings = task.get_settings()
    settings.split_params.set_split_explicit(full, full, dataset_name="ml_train", test_dataset_name="ml_test")
    for column in list(settings.get_raw()["preprocessing"]["per_feature"]):
        if column in lib["FEATURES"]:
            settings.use_feature(column)
            handling = settings.get_feature_preprocessing(column)
            if column in lib["CAT_FEATURES"]:       # one column per value
                handling.update(type="CATEGORY", category_handling="DUMMIFY", missing_handling="NONE",
                                missing_impute_with="MODE", dummy_clip="MAX_NB_CATEGORIES", max_nb_categories=100,
                                dummy_drop="AUTO")
            else:                                   # numbers: missing values = the average, then standardised
                handling.update(type="NUMERIC", numerical_handling="REGULAR", missing_handling="IMPUTE",
                                missing_impute_with="MEAN", rescaling="AVGSTD")
        elif column != "target_scaled":
            settings.reject_feature(column)
    settings.disable_all_algorithms()
    if alpha is None:
        settings.set_algorithm_enabled("LIGHTGBM_REGRESSION", True)
        settings.set_algorithm_enabled("XGBOOST_REGRESSION", True)
        settings.set_metric("RMSE")
    else:
        settings.add_custom_python_model(f"LightGBM quantile {alpha}", QUANTILE_MODEL.format(alpha=alpha))
        settings.set_metric(custom_metric=PINBALL.format(alpha=alpha), custom_metric_greater_is_better=False,
                            custom_metric_name=f"Pinball loss {alpha}")
        settings.get_raw()["modeling"]["metrics"]["customMetrics"].append({
            "name": "Share below prediction", "metricCode": SHARE_BELOW, "greaterIsBetter": False,
            "needsProbability": False, "description": f"Should be close to {alpha}"})
    settings.save()
    task.train(session_name="First training")
    ids = task.get_trained_models_ids()
    if len(ids) == 1:
        return task, ids[0]
    rmse = {}
    for model_id in ids:
        metrics = task.get_trained_model_details(model_id).get_performance_metrics()
        rmse[model_id] = next(v for k, v in metrics.items() if k.lower() == "rmse")
    return task, min(rmse, key=rmse.get)


saved_models = {}
for name, (analysis_name, alpha) in VISUAL_MODELS.items():
    started = time.time()
    task, best = lab_model(analysis_name, alpha)
    deployed = task.deploy_to_flow(best, name, "ml_train", test_dataset="ml_test")
    saved_models[name] = deployed["savedModelId"]
    print(f"{analysis_name}: trained in the Lab and deployed as saved model {name} ({time.time() - started:.0f}s)")

# Evaluate recipes (test on the last 6 months, kept in a model evaluation store) and Score recipes
for name, model_id in saved_models.items():
    store = project.create_model_evaluation_store(f"{name}_evaluations")
    for out in (f"eval_{name}_scored", f"eval_{name}_metrics", f"scored_{name}"):
        new_dataset(out)
    builder = project.new_recipe("evaluation", f"evaluate_{name}")
    builder.with_input_model(model_id)
    builder.with_input("ml_test")
    builder.with_output(f"eval_{name}_scored")
    builder.with_output_metrics(f"eval_{name}_metrics")
    builder.with_output_evaluation_store(store.mes_id)
    evaluate = builder.create()
    builder = project.new_recipe("prediction_scoring", f"score_{name}")
    builder.with_input_model(model_id)
    builder.with_input("ml_scoring")
    builder.with_existing_output(f"scored_{name}")
    score = builder.create()
    for created in (evaluate, score):
        created.compute_schema_updates().apply()
    score.rename(f"score_{name}")         # Dataiku names a Score recipe after its input
    print("recipes", f"evaluate_{name}", f"score_{name}")

for name, (inputs, outputs, _) in FLOW_AFTER_VISUAL.items():
    recipe("python", name, inputs, outputs, RECIPES[name])

zones = {}
def to_zone(zone_name, item):
    if zone_name not in zones:
        zones[zone_name] = flow.create_zone(zone_name)
    zones[zone_name].add_item(item)

for name in RAW:
    to_zone("Raw data", project.get_dataset(name))
for table in lib["TABLES"]:
    to_zone("Staging", project.get_dataset(table))
to_zone("Data prep", project.get_dataset("category_rules"))
for _, outputs, zone_name in list(FLOW.values()) + list(FLOW_AFTER_VISUAL.values()):
    for out in outputs:
        to_zone(zone_name, models if out == "models" else project.get_dataset(out))
for name, model_id in saved_models.items():
    to_zone("Training (visual)", project.get_saved_model(model_id))
    for out in (f"eval_{name}_scored", f"eval_{name}_metrics"):
        to_zone("Training (visual)", project.get_dataset(out))
    to_zone("Inference", project.get_dataset(f"scored_{name}"))
stores = project.list_model_evaluation_stores()
for store in stores:
    to_zone("Training (visual)", store)
print("Flow ready:", ", ".join(zones))


# Production runs. Code models (the models folder), visual saved models and evaluation stores are only rebuilt
# when built by name, so the monthly forecast never retrains by accident.
def build_step(n, label, items, mode):
    return {"id": f"step_{n}", "type": "build_flowitem", "name": label, "enabled": True,
            "runConditionType": "RUN_IF_STATUS_MATCH", "runConditionStatuses": ["SUCCESS", "WARNING"],
            "params": {"builds": [{"type": t, "itemId": i, "partitionsSpec": ""} for t, i in items], "jobType": mode,
                       "proceedOnFailure": False, "handleWarningsAs": "WARNING"}}


datasets = lambda *names: [("DATASET", n) for n in names]
SCENARIOS = {
    "Monthly forecast": (
        "After each warehouse load: check it, then forecast the next 3 months with the current models",
        dict(day=2, hour=7), [
            ("Check the latest warehouse load", datasets("load_status"), "NON_RECURSIVE_FORCED_BUILD"),
            ("Stage, prepare and forecast", datasets("predictions", "predictions_visual", "final_forecast"),
             "RECURSIVE_FORCED_BUILD"),
        ]),
    "Retrain and test the models": (
        "Retrain both ways (code and visual), test them on the last 6 months, compare, and forecast with them",
        dict(day=5, hour=7, repeat_every=3), [
            ("Check the latest warehouse load", datasets("load_status"), "NON_RECURSIVE_FORCED_BUILD"),
            ("Stage and prepare the training data",
             datasets("training_rows", "scoring_rows", "ml_train", "ml_test", "ml_scoring"), "RECURSIVE_FORCED_BUILD"),
            ("Retrain the code models", [("MANAGED_FOLDER", models.id)], "NON_RECURSIVE_FORCED_BUILD"),
            ("Test the code models", datasets("test_results"), "NON_RECURSIVE_FORCED_BUILD"),
            ("Retrain the visual models", [("SAVED_MODEL", m) for m in saved_models.values()],
             "NON_RECURSIVE_FORCED_BUILD"),
            ("Test the visual models", [("MODEL_EVALUATION_STORE", s.mes_id) for s in stores],
             "NON_RECURSIVE_FORCED_BUILD"),
            ("Compare and forecast with the new models", datasets("model_comparison", "final_forecast"),
             "RECURSIVE_BUILD"),
        ]),
}
for name, (description, schedule, steps) in SCENARIOS.items():
    scenario = project.create_scenario(name, "step_based", {
        "active": False, "description": description,
        "params": {"steps": [build_step(n, *step) for n, step in enumerate(steps)]}})
    settings = scenario.get_settings()
    settings.add_monthly_trigger(timezone="Asia/Riyadh", **schedule)
    settings.raw_triggers[-1]["active"] = False    # switch on in production (Automation node)
    try:
        settings.save()
        note = "monthly trigger added, switched off"
    except Exception as e:                         # licences without time-based triggers (Free Edition)
        if "License" not in str(e):
            raise
        note = f"no trigger (not in this licence): add a monthly trigger ({schedule}) on the Automation node"
    print(f"Scenario ready: {name} ({note})")

## 6. First training and forecast

Trains and tests the code models, tests the visual ones, and builds the comparison and the first forecast.

In [ ]:
started = time.time()
steps = [  # first run: what the scenario "Retrain and test the models" does, without redoing the data prep
    ("NON_RECURSIVE_FORCED_BUILD", [(models.id, "MANAGED_FOLDER")]),
    ("RECURSIVE_BUILD", [("test_results", None), ("predictions", None), ("predictions_visual", None)]),
    ("NON_RECURSIVE_FORCED_BUILD", [(s.mes_id, "MODEL_EVALUATION_STORE") for s in stores]),
    ("NON_RECURSIVE_FORCED_BUILD", [("model_comparison", None), ("final_forecast", None)]),
]
for mode, items in steps:
    job = project.new_job(mode).with_auto_update_schema_before_each_recipe_run(True)
    for item, kind in items:
        job.with_output(item, object_type=kind)
    job.start_and_wait()
print(f"Models trained and tested, forecast built in {(time.time() - started) / 60:.1f} minutes")

## 7. Results

`model_comparison`: the code and visual models on the same last 6 months (both trained without them).
`wape` is the EV error as a share of the actual amounts (lower is better); `coverage_p10_p90` should be close
to 0.8. `test_results` has more detail on the code models (per month ahead, and a simple guess to beat).

`final_forecast`: per client, category and month ahead, P10, EV and P90 in SAR, with the obligations
included. `NET` is the client's net cash flow. To switch models, set the project variable `model_source` to
`code` or `visual` and run the scenario Monthly forecast.

In [ ]:
comparison = dataiku.Dataset("model_comparison").get_dataframe()
print("Code vs visual models on the last 6 months (wape: lower is better; coverage: close to 0.8 is best):")
display(comparison[["category_name", "wape_code", "wape_visual", "coverage_p10_p90_code",
                    "coverage_p10_p90_visual", "better_ev"]])

forecast = dataiku.Dataset("final_forecast").get_dataframe()
example = forecast.loc[forecast["status"] == "OK", "client_no"].iloc[0]
print(f"Forecast for client {example} (SAR, models: {MODEL_SOURCE}):")
display(forecast[forecast["client_no"] == example]
        .pivot_table(index="category_name", columns="months_ahead", values=["p10", "ev", "p90"], sort=False)
        .round(0))